In [1]:
from datetime import datetime

import pandas as pd
import yfinance.screener.query

# import pandas as pd

#IMPORTS

from functions import *





In [2]:
#INPUTS

target_ticker = "ADS.DE"

benchmark = "URTH"
start_date = "2020-12-31"
end_date = "2026-08-01"
interval = "1wk"
return_calc = "linear" #linear / log
beta_adjustment = "blume" #blume / vasicek / none
peer_group_beta_method = "median" #average / median
rf_lookback_months = 1 #from valuation date or latest available data
equity_risk_premium = 0.055
size_premium = 0.015
comp_spec_risk_premium = 0.00
target_longt_sp_rating = "A"
debt_spread_lookback_months = 1 #from valuation date or latest available data

peer_group = ["NKE", "PUM.DE", "ONON", "DECK", "CROX"]


# Ce = rf + beta * mrp + sp +csrp
# Cd = rf + debt spread

In [3]:
#CLOSE_DATA_COLLECTION

ticker_package = peer_group + [benchmark]
data_package = download_data(tickers= ticker_package, start_date= start_date, end_date= end_date, interval= interval)
save_data(data = data_package, benchmark= benchmark, peer_group= peer_group, start_date= start_date, end_date= end_date)
close_data = extract_col(data = data_package, field= "Close")



[*********************100%***********************]  6 of 6 completed


In [4]:
#BASIC_DATA_CLEANING
close_data = basic_cleaning(close_data=close_data)

In [5]:
#LOG_RETURN_CALC
return_data = log_return_calc(return_calc=return_calc, close_data= close_data)

In [6]:
#BETA_REGRESSION
beta_results = beta_regression(peer_group=peer_group, return_data=return_data, benchmark=benchmark)

In [7]:
beta_adj = beta_adjustments(beta_results)

In [8]:
beta_res = append_d_e_ratio(beta_results=beta_adj, end_date= end_date)

In [9]:
beta_results = append_tax_rates(beta_results=beta_res)

In [10]:
peer_group_beta = unlevered_beta(beta_adjustment=beta_adjustment, beta_results=beta_results, peer_group_beta_method=peer_group_beta_method)

In [11]:
target_levered_beta = get_target_levered(target=target_ticker, end_date=end_date, peer_group_beta=peer_group_beta)

In [12]:
target_rf_rate = (target_rf(target=target_ticker, BASE_STR_1=BASE_STRING_1, BASE_STR_2=BASE_STRING_2,start_date=start_date, end_date=end_date, rf_lookback=rf_lookback_months)) /100

In [13]:
spread_series = get_rating_spread_series(rating_spread_series=RATING_SPREAD_SERIES, start_date=start_date, end_date=end_date)

In [14]:
interpolated_series = interpolate_spreads(spread_series)

In [15]:
target_spread = get_target_spread(interpolated_series, target_longt_sp_rating, lookback=debt_spread_lookback_months)

In [20]:
return_data

Ticker,CROX,DECK,NKE,ONON,PUM.DE,URTH
Date,,,,,,
2021-09-20,0.007217,-0.106301,-0.043664,-0.075225,-0.017568,0.003227
2021-09-27,-0.097057,-0.059884,-0.016913,-0.153248,-0.016367,-0.023664
2021-10-04,-0.076029,-0.015405,0.036856,-0.014098,0.011709,0.005569
2021-10-11,0.052071,-0.004363,0.036267,-0.011640,0.030965,0.021841
2021-10-18,0.091625,0.058029,0.034618,0.119112,0.014279,0.013282
...,...,...,...,...,...,...
2026-06-29,-0.019411,0.001243,0.081963,-0.006474,0.003358,0.026804
2026-07-06,0.059866,0.012418,0.006351,0.046430,0.047973,0.009771
2026-07-13,0.032761,0.004717,-0.013748,-0.034769,0.017743,-0.013341


In [17]:
a = cumulative_returns(return_data=close_data, return_calc=return_calc)
a

C:\Users\Balint\PycharmProjects\wacccc\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:54: RuntimeWarning: overflow encountered in accumulate
  return bound(*args, **kwds)


Ticker,CROX,DECK,NKE,ONON,PUM.DE,URTH
Date,,,,,,
2021-09-13,1.551800e+02,7.249834e+01,1.442384e+02,3.895000e+01,9.372421e+01,1.202972e+02
2021-09-20,2.456611e+04,4.834578e+03,2.017846e+04,1.477949e+03,8.815706e+03,1.475910e+04
2021-09-27,3.491723e+06,2.993777e+05,2.756661e+06,4.658589e+04,8.073502e+05,1.753935e+06
2021-10-04,4.588125e+08,1.825411e+07,3.903578e+08,1.447454e+06,7.478583e+07,2.095703e+08
2021-10-11,6.340330e+10,1.108238e+09,5.726726e+10,4.446581e+07,7.139685e+09,2.558300e+10
...,...,...,...,...,...,...
2026-06-29,inf,inf,inf,inf,inf,inf
2026-07-06,inf,inf,inf,inf,inf,inf
2026-07-13,inf,inf,inf,inf,inf,inf


In [18]:
wacc = get_wacc(target_ticker=target_ticker, end_date=end_date, rf=target_rf_rate, debt_spread=target_spread, relevered_beta=target_levered_beta, erp=equity_risk_premium, sp=size_premium, csrp=comp_spec_risk_premium)

print(wacc)

0.09528534655231519
